# Turbofan engine demo

## Dataset description

The datasets simulates 100 turbofan engines used in aircraft. The dataset contains time series data for 21 sensors and 3 enviromental conditions:


---
     
1.   Unit: engine number
2.   Time (cycle)
3.   Altitude (feet)
4.   TRA: Throttle resolver angle (deg)
5.   Mach number
6.   T2: Total temperature at fan inlet (ºR - Rankine scale)
7.   T24: Total temperature at LPC outlet (ºR)
8.   T30: Total temperature at HPC outlet (ºR)
9.   T50: Total temperature at LTP outlet (ºR)
10.  P2: Pressure at fan inlet (psia)
11.  P15: Total pressure in bypass conduct (psia)
12.  P30: Total pressure at LPC outlet (psia)
13.  Nf: Physical fan speed (RPM)
14.  Nc: Physical core speed (RPM)
15.  EPR: engine pressure ratio = P50/P2 (-)
16.  Ps30: Static pressure at LPC outlet (psia)
17.  Phi: Ratio of fuel flow to Ps30 (pps/psi)
18.  NRf: Corrected fan speed (RPM)
19.  NRc: Corrected core speed (RPM)
20.  BPR: Bypass ratio (-)
21.  farB: Burner fuel-air ratio (-)
22.  htBleed: Bleed enthalpy (-)
23.  Nf_dmd: Demanded fan speed (RPM)
24.  PCNfR_dmd: Demanded corrected fan speed (RPM)
25.  W31: HPT coolant bleed (lbm/s)
26.  W32: LPT coolant bleed (lbm/s)  


---

The dataset contains four subsets that contain different number of train and test trayectories, operational conditions and fault modes.

For the sake of the demo we'll simulate the train trayectories of one subset. This will imply having 100 engine_ids in one operational mode. We'll simulate this as all engines in the same space_id in the building, like an engine lab.

## Load dataset and upload data

In [ ]:
!git clone https://github.com/andreser09/mha-demo.git

Cloning into 'mha-demo'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 29 (delta 1), reused 29 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 11.97 MiB | 13.65 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [ ]:
cd mha-demo/

In [ ]:
import pandas as pd
from datetime import datetime, timedelta
import requests

In [ ]:
#Get building_id
buildings_url = "http://13.60.16.153/api/v1/buildings"

response = requests.get(buildings_url)

if response.status_code == 200:
    buildings_data = response.json()
    print(buildings_data)
else:
    print(f"Error: Unable to fetch data. Status code: {response.status_code}")

{'buildings': [{'building_id': 'b001', 'name': 'HQ', 'description': 'Headquarters', 'location': '1234 Main St.'}]}


### Create new space: lab002 - Engine lab

In [ ]:
space_body = {
    "space_id": "lab002",
    "building_id": "b001",
    "name": "Engine lab",
    "type": "Lab",
    "floor": 2
}

In [ ]:
spaces_url = "http://13.60.16.153/api/v1/spaces"

response = requests.post(spaces_url, json=space_body)
print(response.json())

{'space': {'building_id': 'b001', 'name': 'Engine lab', 'floor': 2, 'type': 'Lab', 'space_id': 'lab002'}}


In [ ]:
spaces = requests.get(spaces_url)
spaces_data = spaces.json()
print(spaces_data)

{'spaces': [{'building_id': 'b001', 'name': 'Room 0', 'type': 'Operation', 'floor': 1, 'space_id': 'sp1'}, {'building_id': 'b001', 'name': 'Room 1', 'type': 'Operation', 'floor': 1, 'space_id': 'sp2'}, {'building_id': 'b001', 'name': 'Room 2', 'type': 'Operation', 'floor': 1, 'space_id': 'sp3'}, {'building_id': 'b001', 'name': 'Room 3', 'type': 'Operation', 'floor': 1, 'space_id': 'sp4'}, {'building_id': 'b001', 'name': 'Engine lab', 'floor': 2, 'type': 'Lab', 'space_id': 'lab002'}]}


### Prepare engine data for upload

In [ ]:
def create_timestamp(row):
  start_time = datetime(2025, 1, 1, 0, 0, 0)  # Starting time for every engine
  time_delta = timedelta(hours=row['time'])  # Time delta in hours
  timestamp = start_time + time_delta
  return timestamp.strftime("%Y-%m-%dT%H:%M:%SZ")  # Format the timestamp

In [ ]:
columns = ['unit', 'time', 'alt', 'tra', 'mach', 't2', 't24', 't30', 't50', 'p2', 'p15', 'p30', \
           'nf', 'nc', 'epr', 'ps30', 'phi', 'nrf', 'nrc', 'bpr', 'farb', 'htbleed', 'nf_dmd', 'pcnfr_dmd', 'w31', 'w32']
train_1 = pd.read_csv('CMAPSSData/train_FD001.txt', delim_whitespace=True, header=None)
train_1.columns = columns
train_1['equipment_id'] = train_1['unit'].apply(lambda x: 'e000' + str(x) if x < 10 else 'e00' + str(x) if x < 100 else 'e0' + str(x))
train_1['space_id'] = 'lab002'
train_1['timestamp'] = train_1.apply(create_timestamp, axis=1)
train_1.drop(columns=['unit', 'time'], inplace=True)
train_1.head()

<ipython-input-24-9544ad83d432>:3: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  train_1 = pd.read_csv('CMAPSSData/train_FD001.txt', delim_whitespace=True, header=None)


,alt,tra,mach,t2,t24,t30,t50,p2,p15,p30,...,bpr,farb,htbleed,nf_dmd,pcnfr_dmd,w31,w32,equipment_id,space_id,timestamp
0,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,21.61,554.36,...,8.4195,0.03,392,2388,100.0,39.06,23.4190,e0001,lab002,2025-01-01T01:00:00Z
1,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,...,8.4318,0.03,392,2388,100.0,39.00,23.4236,e0001,lab002,2025-01-01T02:00:00Z
2,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,21.61,554.26,...,8.4178,0.03,390,2388,100.0,38.95,23.3442,e0001,lab002,2025-01-01T03:00:00Z
3,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,21.61,554.45,...,8.3682,0.03,392,2388,100.0,38.88,23.3739,e0001,lab002,2025-01-01T04:00:00Z
4,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,21.61,554.00,...,8.4294,0.03,393,2388,100.0,38.90,23.4044,e0001,lab002,2025-01-01T05:00:00Z


In [ ]:
train_1.shape

(20631, 27)

In [ ]:
engine_groups = train_1.groupby('equipment_id')
cycles_per_engine = engine_groups.size()
average_cycles = cycles_per_engine.mean()
print(f"Average number of cycles per engine: {average_cycles}")

Average number of cycles per engine: 206.31


### Upload equipments

In [ ]:
import requests
equipment_ids = train_1['equipment_id'].unique()
equipments_url = "http://13.60.16.153/api/v1/equipment" # Assuming this is your equipments endpoint

for equipment_id in equipment_ids:
    body = {
        "equipment_id": equipment_id,
        "space_id": "lab002",
        "name": "Turbofan Engine",
        "type": "Turbofan",
        "manufacturer": "Pratt&Whitney",
        "model": "Irkut MC-21"
    }
    response = requests.post(equipments_url, json=body)

In [ ]:
equipments = requests.get(equipments_url)
equipments_data = equipments.json()
print(equipments_data)

{'equipment': [{'name': 'Machine 1', 'model': 'model3', 'type': 'Pump', 'space_id': 'sp1', 'equipment_id': '0001', 'manufacturer': 'BM'}, {'name': 'Machine 2', 'model': 'model4', 'type': 'Pump', 'space_id': 'sp1', 'equipment_id': '0002', 'manufacturer': 'BM'}, {'name': 'Machine 3', 'model': 'model3', 'type': 'Pump', 'space_id': 'sp1', 'equipment_id': '0003', 'manufacturer': 'BM'}, {'name': 'Machine 4', 'model': 'model3', 'type': 'Pump', 'space_id': 'sp1', 'equipment_id': '0004', 'manufacturer': 'BM'}, {'name': 'Machine 5', 'model': 'model3', 'type': 'Pump', 'space_id': 'sp1', 'equipment_id': '0005', 'manufacturer': 'BM'}, {'name': 'Machine 6', 'model': 'model3', 'type': 'Pump', 'space_id': 'sp1', 'equipment_id': '0006', 'manufacturer': 'BM'}, {'name': 'Machine 7', 'model': 'model3', 'type': 'Pump', 'space_id': 'sp1', 'equipment_id': '0007', 'manufacturer': 'BM'}, {'name': 'Machine 8', 'model': 'model3', 'type': 'Pump', 'space_id': 'sp1', 'equipment_id': '0008', 'manufacturer': 'BM'}, {

### Upload sensors

In [ ]:
sensor_profiles.keys()

dict_keys(['alt', 'tra', 'mach', 't2', 't24', 't30', 't50', 'p2', 'p15', 'p30', 'nf', 'nc', 'epr', 'ps30', 'phi', 'nrf', 'nrc', 'bpr', 'farb', 'htbleed', 'nf_dmd', 'pcnfr_dmd', 'w31', 'w32'])

In [ ]:
sensor_profiles = {
    "alt": {"type": "altitude", "unit": "feet", "measurement_range": "0-40000"},
    "tra": {"type": "throttle_resolver_angle", "unit": "deg", "measurement_range": "0-100"},
    "mach": {"type": "mach_number", "unit": "-", "measurement_range": "0-1"},
    "t2": {"type": "total_temperature", "unit": "ºR", "measurement_range": "0-1000"},
    "t24": {"type": "total_temperature", "unit": "ºR", "measurement_range": "0-1000"},
    "t30": {"type": "total_temperature", "unit": "ºR", "measurement_range": "0-1000"},
    "t50": {"type": "total_temperature", "unit": "ºR", "measurement_range": "0-1000"},
    "p2": {"type": "pressure", "unit": "psia", "measurement_range": "0-100"},
    "p15": {"type": "pressure", "unit": "psia", "measurement_range": "0-100"},
    "p30": {"type": "pressure", "unit": "psia", "measurement_range": "0-100"},
    "nf": {"type": "physical_fan_speed", "unit": "RPM", "measurement_range": "0-10000"},
    "nc": {"type": "physical_core_speed", "unit": "RPM", "measurement_range": "0-10000"},
    "epr": {"type": "engine_pressure_ratio", "unit": "-", "measurement_range": "0-10"},
    "ps30": {"type": "static_pressure", "unit": "psia", "measurement_range": "0-100"},
    "phi": {"type": "fuel_flow_ratio", "unit": "pps/psi", "measurement_range": "0-1"},
    "nrf": {"type": "corrected_fan_speed", "unit": "RPM", "measurement_range": "0-10000"},
    "nrc": {"type": "corrected_core_speed", "unit": "RPM", "measurement_range": "0-10000"},
    "bpr": {"type": "bypass_ratio", "unit": "-", "measurement_range": "0-10"},
    "farb": {"type": "burner_fuel_air_ratio", "unit": "-", "measurement_range": "0-1"},
    "htbleed": {"type": "bleed_enthalpy", "unit": "-", "measurement_range": "0-10"},
    "nf_dmd": {"type": "demanded_fan_speed", "unit": "RPM", "measurement_range": "0-10000"},
    "pcnfr_dmd": {"type": "demanded_corrected_fan_speed", "unit": "RPM", "measurement_range": "0-10000"},
    "w31": {"type": "hpt_coolant_bleed", "unit": "lbm/s", "measurement_range": "0-10"},
    "w32": {"type": "lpt_coolant_bleed", "unit": "lbm/s", "measurement_range": "0-10"}
}

In [ ]:
def generate_sensor_id(equipment_id, sensor_name):
    return f"{equipment_id}_{sensor_name}"

equipment_ids = train_1['equipment_id'].unique() # assuming your equipment id's are stored in train_1['equipment_id']
all_sensor_ids = []
for equipment_id in equipment_ids:
    for sensor_name in sensor_profiles.keys():
        sensor_id = generate_sensor_id(equipment_id, sensor_name)
        all_sensor_ids.append(sensor_id)

In [ ]:
sensors_url = "http://13.60.16.153/api/v1/sensors"

for equipment_id in equipment_ids:
    for sensor_name in sensor_profiles.keys():
        sensor_id = generate_sensor_id(equipment_id, sensor_name)
        profile = sensor_profiles[sensor_name]
        body = {
            "sensor_id": sensor_id,
            "equipment_id": equipment_id,
            "type": profile["type"],
            "unit": profile["unit"],
            "measurement_range": profile["measurement_range"]
        }
        print(body)
        response = requests.post(sensors_url, json=body)

In [ ]:
equipment_sensor_df = pd.DataFrame(columns=['equipment_id', 'sensor_id'])

for equipment_id in equipment_ids:
    for sensor_name in sensor_profiles.keys():
        sensor_id = generate_sensor_id(equipment_id, sensor_name)
        equipment_sensor_df = pd.concat([equipment_sensor_df, pd.DataFrame({'equipment_id': [equipment_id], 'sensor_id': [sensor_id]})], ignore_index=True)


In [ ]:
equipment_sensor_df.shape

(2400, 2)

In [ ]:
equipment_sensor_df.head()

,equipment_id,sensor_id
0,e0001,e0001_alt
1,e0001,e0001_tra
2,e0001,e0001_mach
3,e0001,e0001_t2
4,e0001,e0001_t24


### Upload data points

In [ ]:
from tqdm import tqdm
import hashlib

In [ ]:
def generate_data_point_id(sensor_id, timestamp):
  # Combine sensor_id and timestamp into a single string
  combined_string = f"{sensor_id}_{timestamp}"

  # Generate SHA-256 hash of the combined string
  hash_object = hashlib.sha256(combined_string.encode('utf-8'))

  # Convert hash to hexadecimal and truncate to 24 characters
  data_point_id = hash_object.hexdigest()[:24]

  return data_point_id


Based on the uploaded sensor_ids and equipment_ids it reamins to batch upload the following data_points

In [ ]:
data_point_body = {
    "data_point_id": generate_data_point_id(sensor_id, timestamp),
    "sensor_id": sensor_id,
    "timestamp": timestamp,
    "value": value
}

In [ ]:
from tqdm import tqdm

data_points = []  # List to store data points
for equipment_id in tqdm(train_1['equipment_id'].unique()):  # Iterate over equipment_ids with progress bar
    for index, row in train_1[train_1['equipment_id'] == equipment_id].iterrows():  # Iterate over rows for the current equipment_id
        timestamp = row['timestamp']
        for sensor_name in sensor_profiles.keys():  # Iterate over sensor names
            sensor_id = generate_sensor_id(equipment_id, sensor_name)
            value = row[sensor_name]

            # Create data point dictionary
            data_point_body = {
                "data_point_id": generate_data_point_id(sensor_id, timestamp),
                "sensor_id": sensor_id,
                "timestamp": timestamp,
                "value": value
            }

            # Add data point to the list
            data_points.append(data_point_body)

# Print the number of data points created:
print(f"Total data points created: {len(data_points)}")

100%|██████████| 100/100 [00:05<00:00, 18.89it/s]

Total data points created: 495144


In [ ]:
import boto3
import json
import os
from multiprocessing import Pool
from tqdm import tqdm

In [ ]:
!mkdir data_points

In [ ]:
def save_data_point(data_point):
    """Saves a single data point as a JSON file to the local folder."""
    data_point_id = data_point['data_point_id']
    file_name = os.path.join('data_points', f"{data_point_id}.json")  # Create file path

    # Convert data point to JSON string and save to file
    with open(file_name, 'w') as f:
        json.dump(data_point, f)

In [ ]:
with Pool(processes=os.cpu_count()) as pool:  # Use all available CPU cores
  list(tqdm(pool.imap(save_data_point, data_points), total=len(data_points)))

100%|██████████| 495144/495144 [02:16<00:00, 3638.78it/s]


In [ ]:
!zip -r data_points.zip data_points